## 0. Configurando sessão spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

:: loading settings :: url = jar:file:/opt/micromamba/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jupyter/.ivy2.5.2/cache
The jars for the packages stored in: /home/jupyter/.ivy2.5.2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6f5f2599-9fd6-4ad4-8243-ed10c4108b57;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 in central
:: resolution report :: resolve 241ms :: artifacts dl 6ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------

In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [3]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [4]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_silver_alfabetizacao_municipio = f"{par_source_project}.silver.meta_alfabetizacao_municipio"
par_source_silver_alfabetizacao_uf = f"{par_source_project}.silver.meta_alfabetizacao_uf"
par_source_silver_alfabetizacao_brasil = f"{par_source_project}.silver.meta_alfabetizacao_brasil"
par_source_gold_rede = f"{par_source_project}.gold.dim_rede"

par_source_gold_fato_meta = f"{par_source_project}.gold.fato_meta"

## 3. Leitura dos dados da origem

In [5]:
df_scr_alfabetizacao_municipio = spark.read.format("bigquery").option("table",par_source_silver_alfabetizacao_municipio).load()
df_scr_alfabetizacao_uf = spark.read.format("bigquery").option("table",par_source_silver_alfabetizacao_uf).load()
df_scr_alfabetizacao_brasil = spark.read.format("bigquery").option("table",par_source_silver_alfabetizacao_brasil).load()
df_scr_rede = spark.read.format("bigquery").option("table",par_source_gold_rede).load()

## 4. Funções auxiliares

In [6]:
def unpivot(df, nivel, local_col):
    local_expr = f"CAST({local_col} AS STRING)" if local_col else "CAST(NULL AS STRING)"
    return (df.selectExpr(
                f"'{nivel}' AS nivel_geografico",
                f"{local_expr} AS local_id",
                "rede",
                "ano AS ano_referencia",
                f"stack({len(ANOS_META)}, {_stack}) AS (ano_meta, valor_meta)")
              .withColumn("ano_meta", F.col("ano_meta").cast("int"))
              .filter(F.col("valor_meta").isNotNull()))

## 5. Transformações

In [7]:
ANOS_META = ["2024","2025","2026","2027","2028","2029","2030"]
_stack = ", ".join([f"'{y}', meta_alfabetizacao_{y}" for y in ANOS_META])

fato_meta = (
    unpivot(df_scr_alfabetizacao_brasil,    "brasil",    None)
    .unionByName(unpivot(df_scr_alfabetizacao_uf,        "uf",        "sigla_uf"))
    .unionByName(unpivot(df_scr_alfabetizacao_municipio, "municipio", "id_municipio"))
    .join(df_scr_rede, on="rede", how="left")
)

## 5. Armazenamento no BQ

In [9]:
(
    fato_meta.write.format("bigquery")
    .option("table", par_source_gold_fato_meta)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)